# 08d · 基础习题集：化学序列分词、语言模型与注意力机制

本习题集配套第 08 讲（`08a` SMILES 语言模型、`08b` 数据增强与迁移微调、`08c` 注意力机制与反应预测 Transformer），共包含 **8 道循序渐进的基础练习**。所有题目均可在无需外部大型数据文件的情况下独立运行，帮助你巩固化学序列建模与 PyTorch 张量操作的核心基本功。

---
## 习题 1：化学感知的正则表达式分词与无损还原检验

在 `08a` 中我们强调过：如果直接用 `list(smiles)` 按单字符切分，溴原子 `Br`、氯原子 `Cl` 以及带电荷/手性的方括号原子团 `[nH]`、`[N+]`、`[O-]` 会被拆碎成毫无化学意义的字母（例如把溴 `Br` 拆成硼 `B` 和芳香碳/环符号）。

请在下方完善函数 `chem_tokenize(smiles)` 与 `verify_roundtrip(smiles_list)`：
1. 使用给定的正则表达式 `TOKEN_PATTERN` 将输入 `smiles` 切分为 token 列表；
2. 对 `test_molecules` 中的每个分子执行分词，再用 `"".join(tokens)` 拼接还原，检验还原后的字符串是否与原 SMILES 完全一致，并返回每个分子的 `(token个数, 单字符长度)` 对比列表。

In [ ]:
import re

TOKEN_PATTERN = re.compile(
    r"(\[[^\]]+]|Br|Cl|Si|Se|@@|@|\(|\)|\.|=|#|-|\+|\\\\|\/|~|[0-9]|[A-Za-z])"
)

test_molecules = [
    "Brc1ccc(-c2scc3cc(Cl)sc23)s1",
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "O=[N+]([O-])c1ccc2[nH]ccc2c1",
]

def chem_tokenize(smiles):
    # TODO: 返回正则匹配出的 token 列表
    pass

def verify_roundtrip(smiles_list):
    # TODO: 检查 "".join(chem_tokenize(s)) == s，返回 [(len(tokens), len(s)), ...]
    pass

# 运行自检
# stats = verify_roundtrip(test_molecules)
# print("Token count vs. Character length:", stats)

---
## 习题 2：起止符封装、长度填充与教师强制（Teacher Forcing）切片

训练自回归序列模型时，我们需要：
1. 在每条 token 序列首尾分别加上起始符 `^`（`BOS`）和终止符 `$`（`EOS`），并用 `<pad>`（索引 `0`）在右侧填充到固定长度 `max_len`；
2. 将编码后的批张量 `X`（形状 `(B, L)`）切分为**输入前缀** `x_in = X[:, :-1]` 与**错位一步的目标序列** `y_target = X[:, 1:]`。

请在下方补全函数 `encode_and_slice_batch(smiles_list, tok2idx, max_len=12)`，返回 `(x_in, y_target)` 两个 `torch.long` 张量。

In [ ]:
import torch

mini_vocab = ["<pad>", "^", "$", "C", "=", "O", "(", ")", "c", "1", "s", "Br"]
mini_tok2idx = {t: i for i, t in enumerate(mini_vocab)}

batch_smiles = ["CC=O", "Brc1cccs1", "CC(=O)C"]

def encode_and_slice_batch(smiles_list, tok2idx, max_len=12):
    # TODO:
    # 1. 对每条 SMILES 分词，前后加 "^" 和 "$"，右侧用 "<pad>" 补齐到 max_len
    # 2. 转为 torch.long 张量 X，返回 X[:, :-1] 和 X[:, 1:]
    pass

# 运行自检
# x_in, y_target = encode_and_slice_batch(batch_smiles, mini_tok2idx, max_len=12)
# print("x_in shape:", x_in.shape, "y_target shape:", y_target.shape)

---
## 习题 3：手写一阶马尔可夫（Bigram）转移概率矩阵

在 `08a` 的第 2 节中，我们首先构建了一个无神经网络的一阶马尔可夫基线模型：统计每个当前 token $x_t$ 之后紧跟下一个 token $x_{t+1}$ 的条件概率分布 $P(x_{t+1} \mid x_t)$。

给定下方由 4 条简单链烃/醚组成的迷你语料库 `toy_corpus`，请补全函数 `build_bigram_matrix(corpus, vocab)`：
1. 对每条分子首尾加上 `^` 和 `$`（无需填充 `<pad>`），统计所有相邻二元组 $(x_t, x_{t+1})$ 的出现频次矩阵 `counts`（形状 `(V, V)`）；
2. 采用拉普拉斯平滑（加 `1e-2` 防止零除），将每一行归一化为概率和等于 `1.0` 的条件概率矩阵 `probs`。

In [ ]:
import numpy as np

toy_vocab = ["^", "$", "C", "O", "="]
toy_corpus = ["CCC", "COCO", "CC=O", "COCC"]

def build_bigram_matrix(corpus, vocab):
    # TODO:
    # 1. 构造 V x V 的频次矩阵 counts
    # 2. 加上 1e-2 平滑后按行归一化，返回 probs
    pass

# 运行自检
# P = build_bigram_matrix(toy_corpus, toy_vocab)
# print("Row sums:", P.sum(axis=1))

---
## 习题 4：追踪 `nn.Embedding + nn.LSTM + nn.Linear` 的张量维度

理解序列模型各层张量的维度变化是调试 PyTorch 代码的关键。给定批大小 $B = 4$、输入序列长度 $T = 10$、词表大小 $V = 20$、词嵌入维度 $E = 16$、双层单向 LSTM（`num_layers = 2`）隐状态维度 $H = 32$：

请在下方补全函数 `trace_lstm_shapes(x)`，依次构建 `emb`、`lstm` 和 `fc` 层，并返回字典记录：
- `"emb_shape"`：嵌入层输出张量的形状；
- `"lstm_out_shape"`：LSTM 各时间步输出张量的形状；
- `"h_n_shape"`：LSTM 最终隐状态 `h_n` 的形状（注意第 0 维是 `num_layers`）；
- `"logits_shape"`：线性输出层 `logits` 的形状。

In [ ]:
import torch
import torch.nn as nn

def trace_lstm_shapes(x, vocab_size=20, emb_dim=16, hid_dim=32, num_layers=2):
    # TODO: 实例化 nn.Embedding, nn.LSTM(batch_first=True), nn.Linear 并返回各步张量的 tuple(shape)
    pass

# 运行自检
# dummy_x = torch.randint(0, 20, (4, 10))
# print(trace_lstm_shapes(dummy_x))

---
## 习题 5：采样温度 $T$ 对 Softmax 概率分布与信息熵的调控

在自回归生成时，我们将模型输出的未归一化对数几率 $\mathbf{z}$（logits）除以温度参数 $T > 0$：

$$P_i(T) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}, \qquad H(P) = -\sum_i P_i(T) \ln P_i(T)$$

请在下方实现函数 `temperature_stats(logits, temperature)`，输入一维张量 `logits` 与标量 `temperature`，返回 `(max_prob, entropy)`（即最大单 token 概率与分布的自然对数香农熵），并验证当 $T$ 从 `0.5` 增大到 `2.0` 时，`max_prob` 单调递减而 `entropy` 单调递增。

In [ ]:
import torch
import torch.nn.functional as F

sample_logits = torch.tensor([2.5, 1.2, 0.3, -0.8, -1.5])

def temperature_stats(logits, temperature):
    # TODO: 计算 probs = softmax(logits / temperature)，返回 ( float(probs.max()), float(entropy) )
    pass

# 运行自检
# for T in [0.5, 1.0, 2.0]:
#     print(f"T={T} -> max_prob={temperature_stats(sample_logits, T)[0]:.4f}, H={temperature_stats(sample_logits, T)[1]:.4f}")

---
## 习题 6：计算每 Token 平均负对数似然（`NLL per token`）并剔除 `<pad>` 干扰

在 `08b` 中，我们使用每 token 平均负对数似然（`NLL per token`）作为无监督探针，衡量一个分子在语言模型眼里是否“自然”（或是否属于分布外样本）。计算时必须将 `ignore_index=pad_id` 传给交叉熵，并且分母只能除以**非 `<pad>` 的真实有效目标 token 数量**。

给定模拟的模型输出 `logits`（形状 `(B, T, V)`）和目标序列 `targets`（形状 `(B, T)`，其中 `0` 代表 `<pad>`），请实现函数 `sequence_nll_per_token(logits, targets, pad_id=0)`，返回形状为 `(B,)` 的每条样本平均 NLL 张量。

In [ ]:
import torch
import torch.nn.functional as F

def sequence_nll_per_token(logits, targets, pad_id=0):
    # TODO:
    # 1. 使用 F.cross_entropy(..., ignore_index=pad_id, reduction="none") 计算逐位置损失
    # 2. 按行求和后除以每行 targets != pad_id 的有效 token 个数
    pass

# 运行自检
# torch.manual_seed(42)
# fake_logits = torch.randn(3, 6, 10)
# fake_targets = torch.tensor([[3, 4, 5, 2, 0, 0], [3, 6, 2, 0, 0, 0], [3, 4, 5, 6, 7, 2]])
# print("Per-sequence NLL:", sequence_nll_per_token(fake_logits, fake_targets, pad_id=0))

---
## 习题 7：手写带因果掩码（Causal Mask）的缩放点积注意力

在 `08c` 中，解码器自注意力层必须施加一个上三角掩码矩阵（`torch.triu(..., diagonal=1)`），使得时间步 $t$ 的 Query 只能关注 $j \le t$ 的历史位置，而对未来位置 $j > t$ 的注意力权重严格为 `0.0`。

请在下方补全函数 `causal_scaled_dot_product_attention(Q, K, V)`，输入形状均为 `(B, T, d_k)` 的张量 `Q, K, V`，返回 `(output, attn_weights)`。

In [ ]:
import math
import torch
import torch.nn.functional as F

def causal_scaled_dot_product_attention(Q, K, V):
    # TODO:
    # 1. 计算 Q @ K^T / sqrt(d_k)
    # 2. 构造上三角布尔掩码 (diagonal=1) 并将对应分数填充为 -inf
    # 3. 执行 softmax 并乘以 V，返回 (output, attn_weights)
    pass

# 运行自检
# q_demo = torch.randn(2, 5, 8)
# out_demo, w_demo = causal_scaled_dot_product_attention(q_demo, q_demo, q_demo)
# print("Upper triangle sum (should be 0.0):", float(torch.triu(w_demo, diagonal=1).sum()))

---
## 习题 8：用 RDKit 反应模板预测 Suzuki 偶联产物并做 Canonical 精确匹配评估

在评估化学反应预测模型（无论是模板法还是 `ReactionTransformer`）时，同一个分子可能写出不同的有效 SMILES 字符串（例如 `c1ccc(-c2cccs2)cc1` 与 `c1csc(-c2ccccc2)c1`）。因此评估 Top-1 准确率时，必须先将预测字符串与真实产物字符串都转换为 **RDKit Canonical SMILES** 再做相等比较。

请在下方实现函数 `run_suzuki_and_check_match(halide_smi, boronic_smi, expected_product_smi)`：
1. 使用 SMARTS 模板 `[c:1][Br,Cl].[c:2]B(O)O>>[c:1]-[c:2]` 对 `(halide_smi, boronic_smi)` 执行反应预测；
2. 将预测得到的第一个产物分子做 `Chem.SanitizeMol` 并转为 Canonical SMILES；
3. 将 `expected_product_smi` 也转为 Canonical SMILES，返回 `(pred_canon, is_exact_match)`。

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdChemReactions

def run_suzuki_and_check_match(halide_smi, boronic_smi, expected_product_smi):
    # TODO:
    # 1. 构建 Suzuki 反应模板并运行 RunReactants
    # 2. 将首个产物与 expected_product_smi 均转为 Canonical SMILES 并比较是否相等
    pass

# 运行自检
# pred_c, matched = run_suzuki_and_check_match("Brc1ccccc1", "OB(O)c1cccs1", "c1ccc(-c2cccs2)cc1")
# print(f"Predicted Canonical: {pred_c} | Exact Match: {matched}")